<a href="https://colab.research.google.com/github/rfroiman/DI_Bootcamp/blob/main/Week9/Day1/Evaluating_LLMs_for_Summarization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Exercises XP : Evaluating LLMs for Summarization



## What you will learn
- Hands-on evaluation for summarization: accuracy vs. ROUGE.
- Strengths/weaknesses of metrics and model size comparisons.
- Using Hugging Face `transformers` + `evaluate` for quick experiments.
- Data loading, sampling, preprocessing, and debugging model outputs.

**Create**: evaluation scripts, comparison tables, custom metrics, and short analyses.


In [1]:

# Part I. Setup (run once per runtime)
# Install minimal deps; keep quiet to reduce noise.
!pip -q install rouge_score==0.1.2 evaluate datasets transformers accelerate nltk --quiet

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True


### Part II. Dataset loading and exploration
Preferred dataset: [abisee/cnn_dailymail](https://huggingface.co/datasets/abisee/cnn_dailymail) (map `article` -> `prompt_text`, `highlights` -> `prompt_title`).
- If you have local train/test CSVs with `prompt_text` / `prompt_title`, set the paths below.
- Otherwise, we will auto-sample a small slice from the HF dataset to keep things light.
- Show a couple of rows for a sanity check.
If HF download fails, a tiny fallback sample is used.


In [2]:

import pandas as pd
from datasets import load_dataset

# Point to your data; leave empty to use the HF cnn_dailymail sample or fallback
train_path = ''  # e.g., '/content/train.csv'
test_path = ''   # e.g., '/content/test.csv'

fallback = pd.DataFrame([
    {
        'prompt_text': 'The cat sat on the mat and purred loudly while the sun set.',
        'prompt_title': 'Cat rests on mat at sunset'
    },
    {
        'prompt_text': 'Scientists discovered water on the moon, opening new research paths.',
        'prompt_title': 'Water found on the moon'
    },
    {
        'prompt_text': 'The local team won the championship after a dramatic final match.',
        'prompt_title': 'Local team clinches title'
    },
])

def load_and_sample(path, split_name, n):
    if path:
        df = pd.read_csv(path)
    else:
        try:
            hf_split = f"{split_name}[:{max(n, 3)}]"
            ds = load_dataset('abisee/cnn_dailymail', '3.0.0', split=hf_split)
            df = ds.to_pandas()[['article', 'highlights']].rename(columns={'article': 'prompt_text', 'highlights': 'prompt_title'})
        except Exception as exc:
            print(f"HF load failed ({exc}); using tiny fallback sample.")
            df = fallback.copy()
    return df.sample(min(n, len(df)), random_state=42).reset_index(drop=True)

train_df = load_and_sample(train_path, 'train', 100)
test_df = load_and_sample(test_path, 'test', 50)

display(train_df.head(2))


README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

3.0.0/train-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00000-of-00003.parquet: downloading bytes:           |  0.00B            

3.0.0/train-00001-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00001-of-00003.parquet: downloading bytes:           |  0.00B            

...

3.0.0/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

,prompt_text,prompt_title
0,"SHANGHAI, China -- Championship leader Lewis H...",Lewis Hamilton fails to clinch world title aft...
1,(CNN) -- China has suspended exports of the Aq...,State-run news agency: China orders an investi...



### Part III. Summarization with T5 (implement)
Tasks:
- Write `batch_generator` to yield mini-batches.
- Write `summarize_with_t5` using `t5-small` (or swap sizes) with GPU if available.
- Prefix inputs with "summarize: " and decode with `skip_special_tokens=True`.
- Clear CUDA cache between batches (`torch.cuda.empty_cache()`) and gc.collect().


In [3]:

import torch
import gc
import pandas as pd

from transformers import AutoTokenizer, T5ForConditionalGeneration
from typing import List


# Generate mini-batches from a list of texts
def batch_generator(items: List[str], batch_size: int):
    for i in range(0, len(items), batch_size):
        yield items[i:i + batch_size]


# Generate summaries using a T5 model
def summarize_with_t5(
    texts: List[str],
    model_name: str = "t5-small",
    batch_size: int = 2,
    max_new_tokens: int = 32
):
    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    print(f"Using device: {device}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = T5ForConditionalGeneration.from_pretrained(model_name)
    model.to(device)
    model.eval()

    summaries = []

    for batch in batch_generator(texts, batch_size):

        inputs = tokenizer(
            ["summarize: " + text for text in batch],
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                num_beams=2
            )

        summaries.extend(
            tokenizer.batch_decode(
                outputs,
                skip_special_tokens=True
            )
        )

        del inputs, outputs

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        gc.collect()

    model.to("cpu")
    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    gc.collect()

    return summaries


# Run a small initial test
RUN_T5 = True

if RUN_T5:
    train_summaries_t5 = summarize_with_t5(
        train_df["prompt_text"].head(5).tolist(),
        model_name="t5-small",
        batch_size=2
    )

    t5_results = pd.DataFrame({
        "prompt_text": train_df["prompt_text"].head(5),
        "reference_summary": train_df["prompt_title"].head(5),
        "t5_small_summary": train_summaries_t5
    })

    display(t5_results)

else:
    print("Skipping T5 generation.")

Using device: cpu


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

,prompt_text,reference_summary,t5_small_summary
0,"SHANGHAI, China -- Championship leader Lewis H...",Lewis Hamilton fails to clinch world title aft...,championship leader Lewis Hamilton spun out of...
1,(CNN) -- China has suspended exports of the Aq...,State-run news agency: China orders an investi...,the toys contain a chemical that converts into...
2,(CNN) -- The company was founded in 1985 by se...,The company has become a huge name in communic...,the company was founded in 1985 by seven commu...
3,"ISLAMABAD, Pakistan (CNN) -- Hours after decla...",NEW: President Musharraf orders troops to take...,president pervez Musharraf orders troops to ta...
4,"QUEBEC, Canada -- Third seed Julia Vakulenko w...",Julia Vakulenko has reached her first final on...,third seed Julia Vakulenko will face comeback ...



### Part IV. Accuracy evaluation (toy, likely near zero)
Implement a naive accuracy that checks exact string match between generated and reference summaries.
Discuss why this is harsh for free-form text (almost always zero).


In [4]:

from typing import List

# Calculate exact-match accuracy
def compute_accuracy(preds: List[str], refs: List[str]) -> float:
    if len(preds) != len(refs):
        raise ValueError("Predictions and references must have the same length.")

    matches = sum(
        1 for p, r in zip(preds, refs)
        if p.strip() == r.strip()
    )

    return matches / len(refs) if refs else 0.0


# Evaluate the five generated summaries
references = train_df["prompt_title"].head(5).tolist()

accuracy = compute_accuracy(
    train_summaries_t5,
    references
)

print(f"Exact-match accuracy: {accuracy:.4f}")

Exact-match accuracy: 0.0000



### Part V. ROUGE metric implementation
Use `evaluate.load("rouge")` and NLTK sentence tokenizer.
Preprocess by joining sentences with newlines for better ROUGE-L.


In [5]:

import evaluate
from nltk.tokenize import sent_tokenize
from typing import List

# Load ROUGE evaluation metric
rouge = evaluate.load("rouge")


# Normalize text for ROUGE evaluation
def normalize_text(text):
    sentences = sent_tokenize(text.strip())
    return "\n".join(sentences)


# Calculate ROUGE scores
def compute_rouge_score(preds: List[str], refs: List[str]):
    if len(preds) != len(refs):
        raise ValueError(
            "Predictions and references must have the same length."
        )

    normalized_preds = [normalize_text(p) for p in preds]
    normalized_refs = [normalize_text(r) for r in refs]

    return rouge.compute(
        predictions=normalized_preds,
        references=normalized_refs,
        use_stemmer=True
    )


# Smoke test
test_preds = ["alpha beta", "", "The cat sat."]
test_refs = ["alpha beta", "reference text", "The cat sat."]

print("ROUGE sanity check:")
print(compute_rouge_score(test_preds, test_refs))

# Evaluate T5-small summaries
print("\nT5-small ROUGE scores:")
t5_rouge_scores = compute_rouge_score(
    train_summaries_t5,
    train_df["prompt_title"].head(5).tolist()
)

print(t5_rouge_scores)

ROUGE sanity check:
{'rouge1': np.float64(0.6666666666666666), 'rouge2': np.float64(0.6666666666666666), 'rougeL': np.float64(0.6666666666666666), 'rougeLsum': np.float64(0.6666666666666666)}

T5-small ROUGE scores:
{'rouge1': np.float64(0.3213216920951289), 'rouge2': np.float64(0.1430145595915109), 'rougeL': np.float64(0.25738069556042925), 'rougeLsum': np.float64(0.3202678772658489)}



### Part VI. Understanding ROUGE scores
Experiments to run (describe your findings in a text cell):
- Exact match vs. empty prediction.
- Effect of stemming: e.g., "running" vs. "run".
- N-gram overlap: see how ROUGE-1 vs. ROUGE-2 change with partial overlap.
- Symmetry: swap preds/refs and compare.


In [6]:
# Part VI - Understanding ROUGE Scores

experiments = [
    ("Exact Match", "The cat sat on the mat.", "The cat sat on the mat."),
    ("Empty Prediction", "", "The cat sat on the mat."),
    ("Stemming", "The dogs are running.", "The dog runs."),
    ("Partial Overlap", "The cat sat on the mat.", "The cat slept on the sofa.")
]

results = []

for name, prediction, reference in experiments:
    scores = compute_rouge_score([prediction], [reference])

    results.append({
        "Experiment": name,
        "ROUGE-1": round(float(scores["rouge1"]), 4),
        "ROUGE-2": round(float(scores["rouge2"]), 4),
        "ROUGE-L": round(float(scores["rougeL"]), 4)
    })

# Test symmetry
forward = compute_rouge_score(
    ["The cat sat on the mat."],
    ["The cat sat."]
)

reverse = compute_rouge_score(
    ["The cat sat."],
    ["The cat sat on the mat."]
)

results.append({
    "Experiment": "Symmetry (Forward)",
    "ROUGE-1": round(float(forward["rouge1"]), 4),
    "ROUGE-2": round(float(forward["rouge2"]), 4),
    "ROUGE-L": round(float(forward["rougeL"]), 4)
})

results.append({
    "Experiment": "Symmetry (Reverse)",
    "ROUGE-1": round(float(reverse["rouge1"]), 4),
    "ROUGE-2": round(float(reverse["rouge2"]), 4),
    "ROUGE-L": round(float(reverse["rougeL"]), 4)
})

rouge_experiments_df = pd.DataFrame(results)

display(rouge_experiments_df)

,Experiment,ROUGE-1,ROUGE-2,ROUGE-L
0,Exact Match,1.0000,1.0000,1.0000
1,Empty Prediction,0.0000,0.0000,0.0000
2,Stemming,0.8571,0.4000,0.8571
3,Partial Overlap,0.6667,0.4000,0.6667
4,Symmetry (Forward),0.6667,0.5714,0.6667
5,Symmetry (Reverse),0.6667,0.5714,0.6667


### Part VI — ROUGE Experiments: Findings

The experiments demonstrate how ROUGE evaluates text overlap between generated and reference summaries.

- **Exact Match:** Identical texts achieved a perfect score of 1.0 across all metrics.
- **Empty Prediction:** An empty prediction received a score of 0.0 because no words matched the reference.
- **Stemming:** The use of stemming improved matching between related word forms, such as "running" and "runs."
- **N-gram Overlap:** ROUGE-1 produced higher scores than ROUGE-2 for partially matching sentences, demonstrating that matching individual words is easier than matching consecutive word pairs.
- **Symmetry:** Swapping predictions and references produced identical F1 scores, showing that ROUGE F1 is symmetric for these examples.

**Conclusion:** ROUGE provides a more informative evaluation of generated summaries than exact-match accuracy. However, ROUGE measures lexical overlap rather than factual correctness or semantic equivalence. Therefore, human evaluation remains important for assessing summary quality.

In [7]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU")

CUDA available: False
Using CPU



### Part VII. Comparing small and large models
Goals:
- Generate summaries with `t5-small`, `t5-base`, and `gpt2` (TL;DR style prompt).
- Compute ROUGE for each and store per-row scores.
- Implement `compute_rouge_per_row` to add ROUGE columns to a DataFrame.
- Implement `summarize_with_gpt2` with a TL;DR: prefix and max length guard.
Use small batches and low `max_new_tokens` to keep things snappy.


In [8]:
# Generate T5-base summaries using the same five articles

comparison_texts = train_df["prompt_text"].head(5).tolist()
comparison_references = train_df["prompt_title"].head(5).tolist()

train_summaries_t5_base = summarize_with_t5(
    comparison_texts,
    model_name="t5-base",
    batch_size=1,
    max_new_tokens=32
)

t5_base_rouge_scores = compute_rouge_score(
    train_summaries_t5_base,
    comparison_references
)

print("T5-base ROUGE scores:")
print(t5_base_rouge_scores)

display(pd.DataFrame({
    "Reference Summary": comparison_references,
    "T5-small Summary": train_summaries_t5,
    "T5-base Summary": train_summaries_t5_base
}))

Using device: cpu


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  892MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

T5-base ROUGE scores:
{'rouge1': np.float64(0.4164102564102564), 'rouge2': np.float64(0.19354925693325006), 'rougeL': np.float64(0.3256739918037138), 'rougeLsum': np.float64(0.3965992983811943)}


,Reference Summary,T5-small Summary,T5-base Summary
0,Lewis Hamilton fails to clinch world title aft...,championship leader Lewis Hamilton spun out of...,championship leader Lewis Hamilton spins out o...
1,State-run news agency: China orders an investi...,the toys contain a chemical that converts into...,china suspends exports of the Aqua Dots toys c...
2,The company has become a huge name in communic...,the company was founded in 1985 by seven commu...,the company was founded in 1985 by seven commu...
3,NEW: President Musharraf orders troops to take...,president pervez Musharraf orders troops to ta...,president pervez muharraf orders troops to tak...
4,Julia Vakulenko has reached her first final on...,third seed Julia Vakulenko will face comeback ...,third seed Julia vakulenko will face former wo...


In [9]:

# Part VII - GPT-2 Summarization and Per-Row ROUGE Evaluation

import torch
import gc
import pandas as pd

from transformers import AutoModelForCausalLM, AutoTokenizer
from typing import List


def summarize_with_gpt2(
    texts: List[str],
    model_name: str = "gpt2",
    batch_size: int = 2,
    max_new_tokens: int = 32
):
    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    print(f"Using device: {device}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(model_name)
    model.config.pad_token_id = tokenizer.eos_token_id
    model.to(device)
    model.eval()

    summaries = []

    for batch in batch_generator(texts, batch_size):

        prompts = [
            text[:1200] + "\nTL;DR:"
            for text in batch
        ]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=768
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        # Decode only the newly generated tokens
        generated_tokens = outputs[:, inputs["input_ids"].shape[1]:]

        summaries.extend(
            tokenizer.batch_decode(
                generated_tokens,
                skip_special_tokens=True
            )
        )

        del inputs, outputs, generated_tokens

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        gc.collect()

    model.to("cpu")
    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    gc.collect()

    return summaries


def compute_rouge_per_row(
    df: pd.DataFrame,
    pred_col: str,
    ref_col: str = "prompt_title"
):
    result = df.copy()

    scores = [
        compute_rouge_score([pred], [ref])
        for pred, ref in zip(result[pred_col], result[ref_col])
    ]

    for metric in ["rouge1", "rouge2", "rougeL", "rougeLsum"]:
        result[f"{pred_col}_{metric}"] = [
            float(score[metric]) for score in scores
        ]

    return result


# Generate GPT-2 summaries using the same five articles
RUN_COMPARE = True

if RUN_COMPARE:
    train_summaries_gpt2 = summarize_with_gpt2(
        comparison_texts,
        model_name="gpt2",
        batch_size=1,
        max_new_tokens=32
    )

    gpt2_rouge_scores = compute_rouge_score(
        train_summaries_gpt2,
        comparison_references
    )

    print("\nGPT-2 ROUGE scores:")
    print(gpt2_rouge_scores)

    comparison_df = pd.DataFrame({
        "prompt_title": comparison_references,
        "t5_small_summary": train_summaries_t5,
        "t5_base_summary": train_summaries_t5_base,
        "gpt2_summary": train_summaries_gpt2
    })

    display(comparison_df)

Using device: cpu


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]


GPT-2 ROUGE scores:
{'rouge1': np.float64(0.31674740162913123), 'rouge2': np.float64(0.0951296838707443), 'rougeL': np.float64(0.22388595060084576), 'rougeLsum': np.float64(0.2717012300611167)}


,prompt_title,t5_small_summary,t5_base_summary,gpt2_summary
0,Lewis Hamilton fails to clinch world title aft...,championship leader Lewis Hamilton spun out of...,championship leader Lewis Hamilton spins out o...,Hamilton's car was caught in the middle of th...
1,State-run news agency: China orders an investi...,the toys contain a chemical that converts into...,china suspends exports of the Aqua Dots toys c...,The Aqua Dots toys contain a chemical that ca...
2,The company has become a huge name in communic...,the company was founded in 1985 by seven commu...,the company was founded in 1985 by seven commu...,Qualcomm is a company that is not only a lead...
3,NEW: President Musharraf orders troops to take...,president pervez Musharraf orders troops to ta...,president pervez muharraf orders troops to tak...,Musharraf's actions are for the good of Pakis...
4,Julia Vakulenko has reached her first final on...,third seed Julia Vakulenko will face comeback ...,third seed Julia vakulenko will face former wo...,Julia Vakulenko will face comeback queen Lind...



### Part VIII. Comparing all models
Implement:
- `compare_models` to aggregate average ROUGE across models.
- `compare_models_summaries` to show side-by-side summaries.
Present the tables and discuss which model wins and why.


In [10]:

# Part VIII - Model Comparison and Evaluation

import pandas as pd


def compare_models(rouge_dict):
    rows = []

    for model_name, scores in rouge_dict.items():
        row = {"Model": model_name}

        for metric in ["rouge1", "rouge2", "rougeL", "rougeLsum"]:
            row[metric] = round(float(scores[metric]), 4)

        rows.append(row)

    return pd.DataFrame(rows).set_index("Model")


def compare_models_summaries(df: pd.DataFrame, pred_cols: list):
    columns = ["prompt_title"] + pred_cols
    return df[columns].copy()


# Aggregate ROUGE scores from the three models
rouge_results = {
    "T5-small": t5_rouge_scores,
    "T5-base": t5_base_rouge_scores,
    "GPT-2": gpt2_rouge_scores
}

# Display aggregate comparison
model_comparison = compare_models(rouge_results)

print("Model Comparison - Aggregate ROUGE Scores:")
display(model_comparison)


# Display generated summaries side by side
summary_comparison = compare_models_summaries(
    comparison_df,
    ["t5_small_summary", "t5_base_summary", "gpt2_summary"]
)

print("\nGenerated Summaries Comparison:")
display(summary_comparison)


# Calculate ROUGE scores for each individual article
per_row_results = comparison_df.copy()

for model_col in [
    "t5_small_summary",
    "t5_base_summary",
    "gpt2_summary"
]:
    per_row_results = compute_rouge_per_row(
        per_row_results,
        pred_col=model_col,
        ref_col="prompt_title"
    )

print("\nPer-Article ROUGE-1 Comparison:")

display(per_row_results[[
    "t5_small_summary_rouge1",
    "t5_base_summary_rouge1",
    "gpt2_summary_rouge1"
]])

Model Comparison - Aggregate ROUGE Scores:


,rouge1,rouge2,rougeL,rougeLsum
Model,,,,
T5-small,0.3213,0.1430,0.2574,0.3203
T5-base,0.4164,0.1935,0.3257,0.3966
GPT-2,0.3167,0.0951,0.2239,0.2717



Generated Summaries Comparison:


,prompt_title,t5_small_summary,t5_base_summary,gpt2_summary
0,Lewis Hamilton fails to clinch world title aft...,championship leader Lewis Hamilton spun out of...,championship leader Lewis Hamilton spins out o...,Hamilton's car was caught in the middle of th...
1,State-run news agency: China orders an investi...,the toys contain a chemical that converts into...,china suspends exports of the Aqua Dots toys c...,The Aqua Dots toys contain a chemical that ca...
2,The company has become a huge name in communic...,the company was founded in 1985 by seven commu...,the company was founded in 1985 by seven commu...,Qualcomm is a company that is not only a lead...
3,NEW: President Musharraf orders troops to take...,president pervez Musharraf orders troops to ta...,president pervez muharraf orders troops to tak...,Musharraf's actions are for the good of Pakis...
4,Julia Vakulenko has reached her first final on...,third seed Julia Vakulenko will face comeback ...,third seed Julia vakulenko will face former wo...,Julia Vakulenko will face comeback queen Lind...



Per-Article ROUGE-1 Comparison:


,t5_small_summary_rouge1,t5_base_summary_rouge1,gpt2_summary_rouge1
0,0.276923,0.557377,0.190476
1,0.202899,0.202899,0.166667
2,0.235294,0.338462,0.289855
3,0.412698,0.444444,0.394366
4,0.482759,0.533333,0.542373



## Wrap-up
- Which metrics felt most informative? Why?
- How did model size impact ROUGE and qualitative quality?
- Where did accuracy break down as a metric?
- How would you extend this to human eval or adversarial probes?
Write a short reflection here.


## Final Reflection and Conclusions

### 1. Which evaluation metrics were most informative?

ROUGE-1, ROUGE-2, and ROUGE-L were more informative than exact-match accuracy. ROUGE-1 measured individual word overlap, ROUGE-2 evaluated consecutive word pairs, and ROUGE-L captured similarities in word sequence.

ROUGE-2 was particularly useful for identifying differences in phrase-level similarity, while ROUGE-L provided additional insight into the structural overlap between summaries.

### 2. How did model size affect summarization performance?

T5-base achieved the highest aggregate scores across all four ROUGE metrics:

| Model | ROUGE-1 | ROUGE-2 | ROUGE-L | ROUGE-Lsum |
|---|---|---|---|---|
| T5-small | 0.3213 | 0.1430 | 0.2574 | 0.3203 |
| T5-base | 0.4164 | 0.1935 | 0.3257 | 0.3966 |
| GPT-2 | 0.3167 | 0.0951 | 0.2239 | 0.2717 |

The larger T5-base model outperformed T5-small on this sample, suggesting that greater model capacity can improve summarization performance.

However, model architecture and training objectives also matter. GPT-2 is a general-purpose language model rather than a model specifically trained for summarization, making direct comparisons more limited.

### 3. Why is exact-match accuracy insufficient?

Exact-match accuracy returned 0.0 for the T5-small summaries, even though several generated summaries contained relevant information.

This happens because exact-match accuracy requires predictions to be identical to their references. Valid summaries can express the same information using different words or sentence structures.

Therefore, exact-match accuracy is too restrictive for evaluating open-ended text generation.

### 4. How could human evaluation improve the assessment?

Human reviewers could assess summary quality using criteria such as:

- Factual accuracy and faithfulness to the original article.
- Coverage of the most important information.
- Clarity, coherence, and readability.
- Conciseness and absence of unnecessary details.

Human evaluation could identify misleading or incorrect statements that lexical-overlap metrics might overlook.

### 5. What adversarial tests could be performed?

Additional experiments could include articles containing contradictory statements, numerical information, similar entity names, negation, and misleading introductory sentences.

These tests would help determine whether models preserve critical facts instead of simply producing fluent summaries.

### 6. Limitations

This experiment evaluated only five CNN/DailyMail articles, using a CPU runtime and a maximum generation length of 32 tokens.

The small sample limits the generalizability of the findings. Additionally, ROUGE measures lexical overlap rather than semantic equivalence or factual correctness.

A more comprehensive evaluation should use a larger test set, multiple random samples, human evaluation, and semantic similarity metrics.

### Final Conclusion

T5-base demonstrated the strongest overall summarization performance in this experiment, followed by T5-small and GPT-2.

The results highlight the importance of selecting evaluation metrics appropriate to the task. Combining ROUGE with human evaluation provides a more reliable assessment of summary quality than relying on exact-match accuracy alone.